In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [10]:
import yfinance as yf

df = yf.download("AAPL", start="2019-01-01", end="2024-01-01")

# Flatten multi-index columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Save cleanly this time
df.to_csv('../data/AAPL_raw.csv')

# Now load normally
df = pd.read_csv('../data/AAPL_raw.csv', index_col='Date', parse_dates=True)
data = df[['Close']].values

print(data.shape)
print(data[:5])

[*********************100%***********************]  1 of 1 completed

(1258, 1)
[[37.50372314]
 [33.76807022]
 [35.20962143]
 [35.13123703]
 [35.80096436]]


In [11]:
scaler=MinMaxScaler(feature_range=(0,1))
scaled_data= scaler.fit_transform(data)

print(f"Min: {scaled_data.min():.4f}, Max: {scaled_data.max():.4f}")

Min: 0.0000, Max: 1.0000


In [12]:
def create_sequences(data , window_size=60):
  X,y=[],[]
  for i in range(window_size,len(data)):
    X.append(data[i - window_size:i,0])
    y.append(data[i,0])
  return np.array(X),np.array(y)

WINDOW_SIZE = 60
X,y=create_sequences(scaled_data,WINDOW_SIZE)

print(f"X shape:{X.shape}")
print(f"y shape:{y.shape}")

X shape:(1198, 60)
y shape:(1198,)


In [13]:
split = int(len(X) * 0.80)

X_train , X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("training:",len(X_train))
print("testing: ",len(X_test))

training: 958
testing:  240


In [15]:
X_train = X_train.reshape((X_train.shape[0] , X_train.shape[1],1))
X_test = X_test.reshape((X_test.shape[0] , X_test.shape[1],1))
print("X_train final shape:",X_train.shape)
print("X_test final shape:",X_test.shape)

X_train final shape: (958, 60, 1)
X_test final shape: (240, 60, 1)


In [16]:
import os
np.save('../data/X_train.npy',X_train)
np.save('../data/X_test.npy',X_test)
np.save('../data/y_train.npy',y_train)
np.save('../data/y_test.npy',y_test)

import joblib
os.makedirs('../models',exist_ok=True)
joblib.dump(scaler,'../models/scaler.pkl')

print("ALL SAVED")

ALL SAVED
